## Expert Knowledge Worker

### A question answering agent that is an expert knowledge worker
### To be used by employees of Insurellm, an Insurance Tech company
### The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

## TODAY:

- Part A: We will divide our documents into CHUNKS
- Part B: We will encode our CHUNKS into VECTORS and put in Chroma
- Part C: We will visualize our vectors

### Notebook structure

Each part is wrapped in **reusable functions**. The loading and chunking steps (Part A) only need to run once. To experiment with a different embedding model, re-run Part B with a new `embeddings` object and then re-run Part C — no need to restart from the top.

### PART A: Divide our documents into chunks

In [1]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_voyageai import VoyageAIEmbeddings
from langchain_cohere import CohereEmbeddings
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [2]:
# price is a factor for our company, so we're going to use a low cost model

MODEL = "gpt-4.1-mini"   # "gpt-4.1-nano"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
nvidia_api_key = os.getenv('NVIDIA_API_KEY')
voyage_api_key = os.getenv('VOYAGEAI_API_KEY')
cohere_api_key = os.getenv('COHERE_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

if nvidia_api_key:
    print(f"NVIDIA API Key exists and begins {nvidia_api_key[:8]}")
else:
    print("NVIDIA API Key not set")

if voyage_api_key:
    print(f"Voyage API Key exists and begins {voyage_api_key[:8]}")
else:
    print("Voyage API Key not set")

if cohere_api_key:
    print(f"Cohere API Key exists and begins {cohere_api_key[:8]}")
else:
    print("Cohere API Key not set")


OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AIzaSyCW
NVIDIA API Key exists and begins nvapi-cS
Voyage API Key exists and begins pa-35Qgl
Cohere API Key exists and begins 6iwDkdOb


### Part A — Function Definitions

The functions below encapsulate each step of loading and chunking the knowledge base. Run this cell once to define them, then call them in the next cell.

In [3]:
def load_knowledge_base(path="knowledge-base/**/*.md"):
    """Load all markdown files and return the concatenated text."""
    files = glob.glob(path, recursive=True)
    print(f"Found {len(files)} files in the knowledge base")

    entire_knowledge_base = ""

    for file_path in files:
        with open(file_path, 'r', encoding='utf-8') as f:
            entire_knowledge_base += f.read()
            entire_knowledge_base += "\n\n"

    print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")
    return entire_knowledge_base


def count_tokens(text, model=MODEL):
    """Count tokens in *text* using the tokenizer for *model*."""
    encoding = tiktoken.encoding_for_model(model)
    tokens = encoding.encode(text)
    token_count = len(tokens)
    print(f"Total tokens for {model}: {token_count:,}")
    return token_count


def load_documents(knowledge_base_dir="knowledge-base"):
    """Load documents from the knowledge base using LangChain's loaders."""
    folders = glob.glob(f"{knowledge_base_dir}/*")

    documents = []
    for folder in folders:
        doc_type = os.path.basename(folder)
        loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
        folder_docs = loader.load()
        for doc in folder_docs:
            doc.metadata["doc_type"] = doc_type
            documents.append(doc)

    print(f"Loaded {len(documents)} documents")
    return documents


def chunk_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into chunks using RecursiveCharacterTextSplitter."""
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = text_splitter.split_documents(documents)

    print(f"Divided into {len(chunks)} chunks")
    print(f"First chunk:\n\n{chunks[0]}")
    return chunks

In [4]:
entire_knowledge_base = load_knowledge_base()
count_tokens(entire_knowledge_base)
documents = load_documents()
chunks = chunk_documents(documents)

Found 76 files in the knowledge base
Total characters in knowledge base: 304,434
Total tokens for gpt-4.1-mini: 63,555
Loaded 76 documents
Divided into 413 chunks
First chunk:

page_content='# Product Summary

# Rellm: AI-Powered Enterprise Reinsurance Solution

## Summary

Rellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.

## Features

### AI-Driven Analytics
Rellm utilizes cutting-edge AI algorithms to provide predictive insights into risk exposures, enabling users to forecast trends and make informed decisions. Its real-tim

In [5]:
documents[0]

Document(metadata={'source': 'knowledge-base/products/Rellm.md', 'doc_type': 'products'}, page_content="# Product Summary\n\n# Rellm: AI-Powered Enterprise Reinsurance Solution\n\n## Summary\n\nRellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.\n\n## Features\n\n### AI-Driven Analytics\nRellm utilizes cutting-edge AI algorithms to provide predictive insights into risk exposures, enabling users to forecast trends and make informed decisions. Its real-time data analysis empowers reinsurance professionals with actionable intellige

In [6]:
documents[1]

Document(metadata={'source': 'knowledge-base/products/Claimllm.md', 'doc_type': 'products'}, page_content="# Product Summary\n\n# Claimllm\n\n## Summary\n\nClaimllm is Insurellm's revolutionary claims processing platform that transforms the claims experience for insurers, adjusters, and policyholders. Powered by advanced AI, machine learning, and computer vision, Claimllm automates claims handling across all insurance lines—from first notice of loss through final settlement. By dramatically reducing processing time, improving accuracy, and enhancing fraud detection, Claimllm enables insurers to deliver exceptional claims service while significantly reducing operational costs. The platform seamlessly integrates with existing policy administration and core systems to create a unified insurance ecosystem.\n\n## Features\n\n### 1. Intelligent FNOL Processing\nClaimllm's AI-powered first notice of loss intake captures claim details through multiple channels including mobile apps, web portal

In [7]:
chunks[50]

Document(metadata={'source': 'knowledge-base/contracts/Contract with GreenField Holdings for Markellm.md', 'doc_type': 'contracts'}, page_content='## Support\n1. **Customer Support Access**: The Client will have access to dedicated support through phone and email during normal business hours to address any inquiries or technical issues.\n2. **Training and Resources**: Provider will offer onboarding training resources to ensure GreenField Holdings can effectively utilize the Markellm platform.\n3. **Performance Reviews**: Quarterly performance reviews will be conducted to analyze platform effectiveness, customer acquisition rates, and marketing strategies, ensuring both parties are aligned on objectives.\n\n## Pricing\n- **Basic Listing Fee**: GreenField Holdings agrees to pay a monthly fee of $199 for a featured listing on the Markellm platform.\n- **Performance-Based Pricing**: An additional fee of $25 per acquired customer lead will be charged, reflecting successful connections made 

In [8]:
chunks[100]

Document(metadata={'source': 'knowledge-base/contracts/Contract with National Claims Network for Claimllm.md', 'doc_type': 'contracts'}, page_content="7. **Business Continuity:** Insurellm provides disaster recovery with 4-hour RTO (Recovery Time Objective) and 1-hour RPO (Recovery Point Objective).\n\n---\n\n## Renewal\n\nThis agreement includes a mutual 120-day renewal notice period. National Claims Network receives guaranteed enterprise pricing for renewal equal to or better than new enterprise customers at renewal time. Contract may be extended in 12-month increments with mutual written agreement.\n\n---\n\n## Features\n\nNational Claims Network will receive the complete Claimllm Enterprise suite:\n\n1. **Unlimited Claims Processing:** No volume restrictions, supporting National's processing of 100,000+ claims annually with scalability to 500,000+ claims as business grows.\n\n2. **White-Label Platform:** Complete branding customization including:\n   - Custom domain names (claims.n

### PART B: Make vectors and store in Chroma

In Week 3, you set up a Hugging Face account and got an HF_TOKEN

At this point, you might want to add it to your `.env` file and run `load_dotenv(override=True)`

(This actually shouldn't be required).

The functions below accept an `embeddings` object, so you can re-run Part B with a different model without touching Part A.

In [9]:
def create_vectorstore(chunks, embeddings, db_name="vector_db"):
    """Embed chunks and store them in a Chroma vector database."""
    if os.path.exists(db_name):
        Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

    vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
    print(f"Vectorstore created with {vectorstore._collection.count()} documents")
    return vectorstore


def inspect_vectorstore(vectorstore):
    """Print the vector count and dimensionality of the store."""
    collection = vectorstore._collection
    count = collection.count()

    sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
    dimensions = len(sample_embedding)
    print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")
    return collection

### Run Part B — choose your embedding model

Swap the `embeddings` line below to experiment with a different model, then re-run this cell and Part C. No need to re-run Part A.

In [34]:
# Pick an embedding model

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

vectorstore = create_vectorstore(chunks, embeddings)
collection = inspect_vectorstore(vectorstore)

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: ed8977e2-34e0-4544-994d-b0967e73cef9)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


Vectorstore created with 413 documents
There are 413 vectors with 384 dimensions in the vector store


### Part C: Visualize!

In [11]:
def prepare_visualization_data(vectorstore):
    """Extract vectors, documents, and metadata from the vectorstore for plotting."""
    collection = vectorstore._collection
    result = collection.get(include=['embeddings', 'documents', 'metadatas'])
    vectors = np.array(result['embeddings'])
    documents = result['documents']
    metadatas = result['metadatas']
    doc_types = [metadata['doc_type'] for metadata in metadatas]
    colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]
    return vectors, documents, doc_types, colors


def visualize_2d(vectors, doc_types, documents, colors):
    """Reduce vectors to 2D with t-SNE and display an interactive scatter plot."""
    tsne = TSNE(n_components=2, random_state=42)
    reduced_vectors = tsne.fit_transform(vectors)

    fig = go.Figure(data=[go.Scatter(
        x=reduced_vectors[:, 0],
        y=reduced_vectors[:, 1],
        mode='markers',
        marker=dict(size=5, color=colors, opacity=0.8),
        text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
        hoverinfo='text'
    )])

    fig.update_layout(title='2D Chroma Vector Store Visualization',
        scene=dict(xaxis_title='x',yaxis_title='y'),
        width=800,
        height=600,
        margin=dict(r=20, b=10, l=10, t=40)
    )

    fig.show()


def visualize_3d(vectors, doc_types, documents, colors):
    """Reduce vectors to 3D with t-SNE and display an interactive 3D scatter plot."""
    tsne = TSNE(n_components=3, random_state=42)
    reduced_vectors = tsne.fit_transform(vectors)

    fig = go.Figure(data=[go.Scatter3d(
        x=reduced_vectors[:, 0],
        y=reduced_vectors[:, 1],
        z=reduced_vectors[:, 2],
        mode='markers',
        marker=dict(size=5, color=colors, opacity=0.8),
        text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
        hoverinfo='text'
    )])

    fig.update_layout(
        title='3D Chroma Vector Store Visualization',
        scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
        width=900,
        height=700,
        margin=dict(r=10, b=10, l=10, t=40)
    )

    fig.show()

In [12]:
vectors, doc_texts, doc_types, colors = prepare_visualization_data(vectorstore)
visualize_2d(vectors, doc_types, doc_texts, colors)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [13]:
visualize_3d(vectors, doc_types, doc_texts, colors)

In [31]:
# Pick an embedding model

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L12-v2")
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

vectorstore = create_vectorstore(chunks, embeddings)
collection = inspect_vectorstore(vectorstore)

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 66e9ac2b-9412-4f39-8545-0c6ef0d1b9f1)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L12-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vectorstore created with 413 documents
There are 413 vectors with 384 dimensions in the vector store


In [32]:
vectors, doc_texts, doc_types, colors = prepare_visualization_data(vectorstore)
visualize_2d(vectors, doc_types, doc_texts, colors)

In [33]:
visualize_3d(vectors, doc_types, doc_texts, colors)

### OpenAI Embeddings

In [14]:
# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = create_vectorstore(chunks, embeddings)
collection = inspect_vectorstore(vectorstore)

Vectorstore created with 413 documents
There are 413 vectors with 1,536 dimensions in the vector store


In [15]:
vectors, doc_texts, doc_types, colors = prepare_visualization_data(vectorstore)
visualize_2d(vectors, doc_types, doc_texts, colors)

In [16]:
visualize_3d(vectors, doc_types, doc_texts, colors)

In [17]:
# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

vectorstore = create_vectorstore(chunks, embeddings)
collection = inspect_vectorstore(vectorstore)

Vectorstore created with 413 documents
There are 413 vectors with 3,072 dimensions in the vector store


In [18]:
vectors, doc_texts, doc_types, colors = prepare_visualization_data(vectorstore)
visualize_2d(vectors, doc_types, doc_texts, colors)

In [19]:
visualize_3d(vectors, doc_types, doc_texts, colors)

### Gemini Embeddings

In [20]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

vectorstore = create_vectorstore(chunks, embeddings)
collection = inspect_vectorstore(vectorstore)

Vectorstore created with 413 documents
There are 413 vectors with 3,072 dimensions in the vector store


In [21]:
vectors, doc_texts, doc_types, colors = prepare_visualization_data(vectorstore)
visualize_2d(vectors, doc_types, doc_texts, colors)

In [22]:
visualize_3d(vectors, doc_types, doc_texts, colors)

### Nvidia llama Embeddings

In [23]:
embeddings = NVIDIAEmbeddings(
  model="nvidia/llama-nemotron-embed-vl-1b-v2",
  api_key=nvidia_api_key,
  truncate="NONE",
  )

vectorstore = create_vectorstore(chunks, embeddings)
collection = inspect_vectorstore(vectorstore)

/Users/patrickwalukagga/Projects/personal/llm_engineering/.venv/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:243: UserWarning:

Found nvidia/llama-nemotron-embed-vl-1b-v2 in available_models, but type is unknown and inference may fail.



Vectorstore created with 413 documents
There are 413 vectors with 2,048 dimensions in the vector store


In [24]:
vectors, doc_texts, doc_types, colors = prepare_visualization_data(vectorstore)
visualize_2d(vectors, doc_types, doc_texts, colors)

In [25]:
visualize_3d(vectors, doc_types, doc_texts, colors)

### Voyiage Embeddings

In [27]:
try:
    embeddings = VoyageAIEmbeddings(
        voyage_api_key=voyage_api_key, model="voyage-4-large"
    )
    vectorstore = create_vectorstore(chunks, embeddings)
    collection = inspect_vectorstore(vectorstore)
except Exception as e:
    print(f"Error creating vectorstore: {e}")


tokenizer.json: 0.00B [00:00, ?B/s]

Error creating vectorstore: You have not yet added your payment method in the billing page and will have reduced rate limits of 3 RPM and 10K TPM. To unlock our standard rate limits, please add a payment method in the billing page for the appropriate organization in the user dashboard (https://dashboard.voyageai.com/). Even with payment methods entered, the free tokens (200M tokens for Voyage series 3) will still apply. After adding a payment method, you should see your rate limits increase after several minutes. See our pricing docs (https://docs.voyageai.com/docs/pricing) for the free tokens for your model.


In [ ]:
vectors, doc_texts, doc_types, colors = prepare_visualization_data(vectorstore)
visualize_2d(vectors, doc_types, doc_texts, colors)


In [ ]:
visualize_3d(vectors, doc_types, doc_texts, colors)

### Cohere Embeddings

In [28]:
embeddings = CohereEmbeddings(
    model="embed-english-v3.0",
)

vectorstore = create_vectorstore(chunks, embeddings)
collection = inspect_vectorstore(vectorstore)

Vectorstore created with 413 documents
There are 413 vectors with 1,024 dimensions in the vector store


In [29]:
vectors, doc_texts, doc_types, colors = prepare_visualization_data(vectorstore)
visualize_2d(vectors, doc_types, doc_texts, colors)


In [30]:
visualize_3d(vectors, doc_types, doc_texts, colors)